In [29]:
from __future__ import annotations

import copy
import csv
import json
import re
from pathlib import Path
from typing import Any

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import numpy as np
from scipy.ndimage import zoom
from scipy.spatial import cKDTree
from tqdm import tqdm

In [27]:
import h5py
import numpy as np
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader

class BraTSSliceDataset(Dataset):
    def __init__(self, files: list[Path]) -> None:
        self.files = list(files)

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor, str]:
        path = self.files[index]

        with h5py.File(path, "r") as h5_file:
            image = np.asarray(h5_file["image"][...], dtype=np.float32)
            mask = np.asarray(h5_file["mask"][...], dtype=np.float32)

        # 1. Image normalization
        scale = np.max(np.abs(image), axis=(0, 1), keepdims=True)
        scale[scale < 1e-6] = 1.0
        image = np.clip(image / scale, -1.0, 1.0)
        image_tensor = torch.from_numpy(image).permute(2, 0, 1)

        # 2. Fix Mask Shape for CrossEntropyLoss
        # If mask is stored as [H, W, C] (multi-channel binary or one-hot):
        if mask.ndim == 3:
            # Convert multi-channel binary mask to a single class index map [H, W]
            # Example: argmax across channels gives class indices 0, 1, 2...
            mask = np.argmax(mask, axis=-1).astype(np.int64)
        else:
            mask = mask.astype(np.int64)

        # Resulting mask tensor shape: [H, W] with integer class labels
        mask_tensor = torch.from_numpy(mask)

        return image_tensor, mask_tensor, str(path)

In [21]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

class BraTSResNetLatent(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.encoder = resnet50(weights=ResNet50_Weights.DEFAULT)
        
        # 1. Modify first conv layer for 4 MRI modalities
        old_conv = self.encoder.conv1
        self.encoder.conv1 = nn.Conv2d(
            in_channels=4, 
            out_channels=old_conv.out_channels, 
            kernel_size=old_conv.kernel_size, 
            stride=old_conv.stride, 
            padding=old_conv.padding, 
            bias=False
        )
        
        # Initialize the 4th channel weights smoothly from the 3-channel weights
        with torch.no_grad():
            self.encoder.conv1.weight[:, :3, :, :] = old_conv.weight
            self.encoder.conv1.weight[:, 3:4, :, :] = old_conv.weight.mean(dim=1, keepdim=True)

        # 2. Keep and resize the fully connected layer
        in_features = self.encoder.fc.in_features
        self.encoder.fc = nn.Linear(in_features, latent_dim)

    def forward(self, x):
        return self.encoder(x)

# Example usage for a BraTS slice [Batch=2, Channels=4, Height=240, Width=240]
brats_model = BraTSResNetLatent(latent_dim=128)
mri_slice = torch.randn(2, 4, 240, 240)
latent_vector = brats_model(mri_slice)
print("BraTS Latent Shape:", latent_vector.shape) # torch.Size([2, 128])

BraTS Latent Shape: torch.Size([2, 128])


In [22]:
data_dir = Path("data/3/BraTS2020_training_data/content/data")  # Update with your actual directory
h5_files = list(data_dir.glob("*.h5"))

# Create dataset and data loader
dataset = BraTSSliceDataset(files=h5_files)
dataloader = DataLoader(
    dataset, 
    batch_size=8,     # Adjust based on your GPU VRAM
    shuffle=True, 
    num_workers=4     # Utilizes multi-processing with h5py safely
)

# 2. Instantiate the model and move to GPU (if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BraTSResNetLatent(latent_dim=128)
model.to(device)
model.eval()  # Set to eval mode for feature extraction (disables Dropout/BatchNorm updates)

# 3. Iterate through the DataLoader and extract latent vectors
with torch.no_grad():
    for batch_idx, (images, masks, paths) in enumerate(dataloader):
        # Move images to the device
        images = images.to(device)  # Shape: [Batch, 4, H, W]
        masks = masks.to(device)    # Shape: [Batch, H, W] or [Batch, C, H, W]
        
        # Forward pass through the model
        latent_vectors = model(images)  # Shape: [Batch, 128]
        
        print(f"--- Batch {batch_idx + 1} ---")
        print(f"Images input shape:  {images.shape}")
        print(f"Latent space shape:  {latent_vectors.shape}")
        print(f"Masks shape:         {masks.shape}")
        print(f"Sample file paths:   {paths[:2]}") # Print first two paths in batch
        
        break  # Stopping after one b

--- Batch 1 ---
Images input shape:  torch.Size([8, 4, 240, 240])
Latent space shape:  torch.Size([8, 128])
Masks shape:         torch.Size([8, 3, 240, 240])
Sample file paths:   ('data/3/BraTS2020_training_data/content/data/volume_147_slice_146.h5', 'data/3/BraTS2020_training_data/content/data/volume_223_slice_9.h5')


In [33]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.models import ResNet50_Weights
from torchvision.models.segmentation import fcn_resnet50
from pathlib import Path

# --- 1. Model Setup (Spatial Feature Backbone + Segmentation Head) ---
num_classes = 4  # Background + 3 BraTS tumor sub-regions
model = fcn_resnet50(weights=None, weights_backbone=ResNet50_Weights.DEFAULT, num_classes=num_classes)

# Modify first conv layer to accept 4 MRI channels (FLAIR, T1, T1ce, T2)
old_conv = model.backbone.conv1
new_conv = nn.Conv2d(
    in_channels=4, 
    out_channels=old_conv.out_channels, 
    kernel_size=old_conv.kernel_size, 
    stride=old_conv.stride, 
    padding=old_conv.padding, 
    bias=False
)

with torch.no_grad():
    new_conv.weight[:, :3, :, :] = old_conv.weight
    new_conv.weight[:, 3:4, :, :] = old_conv.weight.mean(dim=1, keepdim=True)

model.backbone.conv1 = new_conv

# Update classifier heads for your specific number of classes
model.classifier[4] = nn.Conv2d(model.classifier[4].in_channels, num_classes, kernel_size=1)
if model.aux_classifier is not None:
    model.aux_classifier[4] = nn.Conv2d(model.aux_classifier[4].in_channels, num_classes, kernel_size=1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# --- 2. Dataset and DataLoader Setup ---

dataset = BraTSSliceDataset(files=train_files)
dataloader = DataLoader(
    dataset, 
    batch_size=4,     # Adjust based on your GPU VRAM
    shuffle=True, 
    num_workers=2
)

# --- 3. Loss Function & Optimizer ---
# CrossEntropyLoss expects masks to be integer class indices [Batch, Height, Width]
criterion = nn.CrossEntropyLoss(ignore_index=255) 
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)

# --- 4. Training Loop ---
num_epochs = 10

for epoch in range(num_epochs):
    model.train()  # Switch to training mode (enables BatchNorm updates & dropout)
    epoch_loss = 0.0
    
    for images, masks, paths in dataloader:
        images = images.to(device)         # Shape: [Batch, 4, H, W]
        masks = masks.to(device).long()    # Shape: [Batch, H, W] containing class indices
        
        optimizer.zero_grad()
        
        # Forward pass (Torchvision segmentation models return an OrderedDict)
        outputs = model(images)
        output = outputs['out']            # Shape: [Batch, num_classes, H, W]
        
        # Calculate main segmentation loss
        loss = criterion(output, masks)
        
        # Include auxiliary loss if present (helps stabilize deep backbone training)
        if model.aux_classifier is not None:
            aux_output = outputs['aux']
            loss += 0.5 * criterion(aux_output, masks)
            
        # Backward pass & gradient step
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_epoch_loss = epoch_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{num_epochs}] | Training Loss: {avg_epoch_loss:.4f}")

# Save the fine-tuned model checkpoint
torch.save(model.state_dict(), "brats_resnet_segmentation.pth")
print("Model fine-tuned and saved successfully!")

Epoch [1/10] | Training Loss: 0.0154
Epoch [2/10] | Training Loss: 0.0075
Epoch [3/10] | Training Loss: 0.0063
Epoch [4/10] | Training Loss: 0.0056
Epoch [5/10] | Training Loss: 0.0053
Epoch [6/10] | Training Loss: 0.0050
Epoch [7/10] | Training Loss: 0.0048
Epoch [8/10] | Training Loss: 0.0046
Epoch [9/10] | Training Loss: 0.0045
Epoch [10/10] | Training Loss: 0.0044
Model fine-tuned and saved successfully!


In [49]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from pathlib import Path

def evaluate_segmentation_metrics(
    model: torch.nn.Module,
    val_files: list[Path],
    device: torch.device,
    num_classes: int = 4  # Background + BraTS tumor sub-regions (e.g., NCR, ED, ET)
):
    """
    Evaluates the model on validation files and computes mean Dice and IoU scores.
    """
    model.eval()
    val_dataset = BraTSSliceDataset(files=val_files)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)

    # Track intersection and union counts per class across all slices
    # Shape: [num_classes]
    total_intersection = np.zeros(num_classes, dtype=np.float64)
    total_union = np.zeros(num_classes, dtype=np.float64)
    total_target_area = np.zeros(num_classes, dtype=np.float64)
    total_pred_area = np.zeros(num_classes, dtype=np.float64)

    print(f"Running evaluation on {len(val_files)} validation files...")

    with torch.no_grad():
        for images, masks, paths in val_loader:
            images = images.to(device)
            masks = masks.to(device).long() # Shape: [Batch, H, W]

            # Forward pass (Assuming UNet returns logits as the first element)
            outputs = model(images)
            logits = outputs[0] if isinstance(outputs, tuple) else outputs

            # Convert logits to predicted class indices: [Batch, H, W]
            preds = torch.argmax(logits["out"], dim=1)

            # Move tensors to numpy for metric calculations
            preds_np = preds.cpu().numpy()
            masks_np = masks.cpu().numpy()

            for c in range(num_classes):
                pred_c = (preds_np == c)
                mask_c = (masks_np == c)

                intersection = np.logical_and(pred_c, mask_c).sum()
                union = np.logical_or(pred_c, mask_c).sum()

                total_intersection[c] += intersection
                total_union[c] += union
                total_target_area[c] += mask_c.sum()
                total_pred_area[c] += pred_c.sum()

    # Calculate Dice and IoU per class
    # Formula: Dice = (2 * Intersection) / (Pred_Area + Target_Area)
    # Formula: IoU  = Intersection / Union
    class_dice = np.zeros(num_classes)
    class_iou = np.zeros(num_classes)

    print("\n--- Per-Class Evaluation Results ---")
    for c in range(num_classes):
        denom_dice = total_pred_area[c] + total_target_area[c]
        class_dice[c] = (2.0 * total_intersection[c] / denom_dice) if denom_dice > 0 else 1.0

        class_iou[c] = (total_intersection[c] / total_union[c]) if total_union[c] > 0 else 1.0
        
        print(f"Class {c}: Dice = {class_dice[c]:.4f} | IoU = {class_iou[c]:.4f}")

    # Compute Macro-Averages (ignoring background class index 0 if preferred, or averaging all)
    mean_dice = np.mean(class_dice[1:])  # Averaging tumor classes (1, 2, 3...)
    mean_iou = np.mean(class_iou[1:])

    print(f"\nMean Tumor Dice Score: {mean_dice:.4f}")
    print(f"Mean Tumor IoU Score:  {mean_iou:.4f}")

    return {
        "class_dice": class_dice,
        "class_iou": class_iou,
        "mean_dice": mean_dice,
        "mean_iou": mean_iou
    }

In [51]:
num_epochs = 10
val_dataset = BraTSSliceDataset(files=val_files)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)
num_epochs = 10

model.eval()
val_loss = 0.0
for epoch in range(num_epochs):
    with torch.no_grad():
        for images, masks, paths in val_loader:
            images, masks = images.to(device), masks.to(device).long()
            
            outputs = model(images)
            loss = criterion(outputs['out'], masks)
            val_loss += loss.item()
            
        avg_val_loss = val_loss / len(val_loader)
    
        print(f"Epoch [{epoch+1}/{num_epochs}] | Val Loss: {avg_val_loss:.4f}")
        # Run evaluation metrics on validation set every epoch (or every N epochs)
        eval_results = evaluate_segmentation_metrics(model, val_files, device, num_classes=4)

Epoch [1/10] | Val Loss: 0.0049
Running evaluation on 8579 validation files...

--- Per-Class Evaluation Results ---
Class 0: Dice = 0.9991 | IoU = 0.9981
Class 1: Dice = 0.8856 | IoU = 0.7947
Class 2: Dice = 0.8428 | IoU = 0.7283
Class 3: Dice = 1.0000 | IoU = 1.0000

Mean Tumor Dice Score: 0.9095
Mean Tumor IoU Score:  0.8410
Epoch [2/10] | Val Loss: 0.0098
Running evaluation on 8579 validation files...

--- Per-Class Evaluation Results ---
Class 0: Dice = 0.9991 | IoU = 0.9981
Class 1: Dice = 0.8856 | IoU = 0.7947
Class 2: Dice = 0.8428 | IoU = 0.7283
Class 3: Dice = 1.0000 | IoU = 1.0000

Mean Tumor Dice Score: 0.9095
Mean Tumor IoU Score:  0.8410
Epoch [3/10] | Val Loss: 0.0147
Running evaluation on 8579 validation files...

--- Per-Class Evaluation Results ---
Class 0: Dice = 0.9991 | IoU = 0.9981
Class 1: Dice = 0.8856 | IoU = 0.7947
Class 2: Dice = 0.8428 | IoU = 0.7283
Class 3: Dice = 1.0000 | IoU = 1.0000

Mean Tumor Dice Score: 0.9095
Mean Tumor IoU Score:  0.8410
Epoch [4/1

In [54]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from pathlib import Path

def test_model_pipeline(
    model: torch.nn.Module,
    test_files: list[Path],
    device: torch.device,
    num_classes: int = 4  # Background, NCR/NET, ED, ET
):
    """
    Evaluates the model on a dedicated test dataset, computing per-class 
    and mean Dice/IoU scores along with qualitative predictions.
    """
    model.eval()
    
    # 1. Initialize test loader (shuffle=False for deterministic evaluation order)
    test_dataset = BraTSSliceDataset(files=test_files)
    test_loader = DataLoader(
        test_dataset, 
        batch_size=1, 
        shuffle=False, 
        num_workers=2
    )

    # Accumulators for global confusion metrics per class
    total_intersection = np.zeros(num_classes, dtype=np.float64)
    total_union = np.zeros(num_classes, dtype=np.float64)
    total_target_area = np.zeros(num_classes, dtype=np.float64)
    total_pred_area = np.zeros(num_classes, dtype=np.float64)

    print(f"Running inference and evaluation on {len(test_files)} test files...")

    results_log = []

    with torch.inference_mode():
        for batch_idx, (images, masks, paths) in enumerate(test_loader):
            images = images.to(device)
            masks = masks.to(device).long()  # Shape: [1, H, W]
            file_name = Path(paths[0]).name

            # Forward pass (Unpacking logits if model returns a tuple/dict)
            outputs = model(images)
            if isinstance(outputs, (dict, tuple)):
                logits = outputs["out"]
            else:
                logits = outputs

            # Argmax to get predicted class mask: [1, H, W] -> [H, W]
            preds = torch.argmax(logits, dim=1)

            preds_np = preds.cpu().numpy()[0]
            masks_np = masks.cpu().numpy()[0]

            # Compute slice-level metrics
            slice_dice = {}
            for c in range(num_classes):
                pred_c = (preds_np == c)
                mask_c = (masks_np == c)

                intersection = np.logical_and(pred_c, mask_c).sum()
                union = np.logical_or(pred_c, mask_c).sum()
                
                target_sum = mask_c.sum()
                pred_sum = pred_c.sum()

                total_intersection[c] += intersection
                total_union[c] += union
                total_target_area[c] += target_sum
                total_pred_area[c] += pred_sum

                # Dice for this class on this specific slice
                denom = pred_sum + target_sum
                slice_dice[c] = (2.0 * intersection / denom) if denom > 0 else 1.0

            results_log.append({
                "file": file_name,
                "slice_dice": slice_dice,
                "pred_mask": preds_np,
                "true_mask": masks_np
            })

    # --- Compute Global Test Metrics ---
    class_dice = np.zeros(num_classes)
    class_iou = np.zeros(num_classes)

    print("\n" + "="*40)
    print("FINAL TEST SET EVALUATION RESULTS")
    print("="*40)
    
    class_names = ["Background (0)", "NCR/NET (1)", "Edema (2)", "Enhancing Tumor (3/4)"]
    
    for c in range(num_classes):
        denom_dice = total_pred_area[c] + total_target_area[c]
        class_dice[c] = (2.0 * total_intersection[c] / denom_dice) if denom_dice > 0 else 1.0
        class_iou[c] = (total_intersection[c] / total_union[c]) if total_union[c] > 0 else 1.0
        
        print(f"{class_names[c]:<25} | Dice: {class_dice[c]:.4f} | IoU: {class_iou[c]:.4f}")

    # Mean tumor score (excluding background class index 0)
    mean_dice = np.mean(class_dice[1:])
    mean_iou = np.mean(class_iou[1:])

    print("-" * 40)
    print(f"{'Mean Tumor Score':<25} | Dice: {mean_dice:.4f} | IoU: {mean_iou:.4f}")
    print("="*40)

    return {
        "class_dice": class_dice,
        "class_iou": class_iou,
        "mean_dice": mean_dice,
        "mean_iou": mean_iou,
        "detailed_results": results_log
    }

In [55]:
# 2. Execute test evaluation on your held-out test file list
test_results = test_model_pipeline(
    model=model,
    test_files=test_files,
    device=device,
    num_classes=4
)

Running inference and evaluation on 8580 test files...

FINAL TEST SET EVALUATION RESULTS
Background (0)            | Dice: 0.9990 | IoU: 0.9981
NCR/NET (1)               | Dice: 0.8857 | IoU: 0.7949
Edema (2)                 | Dice: 0.8459 | IoU: 0.7330
Enhancing Tumor (3/4)     | Dice: 1.0000 | IoU: 1.0000
----------------------------------------
Mean Tumor Score          | Dice: 0.9106 | IoU: 0.8426


In [32]:
import random
from pathlib import Path
from torch.utils.data import DataLoader

# 1. Gather all your HDF5 files
data_dir = Path("data/3/BraTS2020_training_data/content/data")
h5_files = list(data_dir.glob("*.h5"))

# Optional: If your filenames contain patient IDs (e.g., "BraTS20_Training_001_slice12.h5"),
# group them by patient first to ensure no patient overlap between sets.

# 2. Shuffle and Split (e.g., 70% Train, 15% Validation, 15% Test)
random.seed(42)  # For reproducibility
random.shuffle(h5_files)

total_files = len(h5_files)
train_split = int(0.70 * total_files)
val_split = int(0.15 * total_files)

train_files = h5_files[:train_split]
val_files = h5_files[train_split:train_split + val_split]
test_files = h5_files[train_split + val_split:]

print(f"Total files: {total_files}")
print(f"Train set: {len(train_files)} files")
print(f"Val set:   {len(val_files)} files")
print(f"Test set:  {len(test_files)} files")

# 3. Create separate Dataset and DataLoader instances for each split
train_dataset = BraTSSliceDataset(files=train_files)
val_dataset = BraTSSliceDataset(files=val_files)
test_dataset = BraTSSliceDataset(files=test_files)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

Total files: 57195
Train set: 40036 files
Val set:   8579 files
Test set:  8580 files
